# DACFE_Ex: Apparent engineering properties
#### **(by Norbert Blanco)**

This notebook computes the apparent engineering properties of a symmetric and balanced composite laminate.

Converted from the original MATLAB Live Script (`DACFE_Ex201.m`) to Python. It runs in Jupyter and in Google Colab without any additional setup (only `numpy` and `pandas`, both preinstalled in Colab).

## Initialization

Import the packages used throughout the notebook. `numpy` handles the linear algebra and `pandas` is used to display the final results as a formatted table.

In [ ]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=4, suppress=False)


## Definition of material and LSS

#### Definition of material properties

- The material is assumed to be transversely isotropic, so only 5 elastic constants are required: E11 (MPa), E22 (MPa), nu12, nu23, G12 (MPa).
- CFRP T300/5208 is defined as material 1. Additional materials can be defined following the same structure `Mat[mat_num] = [E11, E22, nu12, nu23, G12]`, or by updating the material properties.

The material matrix is stored as a dictionary keyed by material number (1, 2, 3, ...) to mirror the row indexing used in the original MATLAB script.

In [ ]:
# Mat[mat_num] = [E11, E22, nu12, nu23, G12]
Mat = {
    1: [181000, 10300, 0.28, 0.42, 7170],
}


#### Definition of the Laminate Stacking Sequence

- The LSS is defined through material (previously defined), orientation (in degrees) and thickness (in mm).
- The first layer is assumed to be the one on the bottom, according to the Z direction.
- Define as many layers as required following this structure: `L[layer_num] = [mat_num, orientation, thickness]`.
- If the laminate is symmetric (as it should be), set `LSS_symm = True` and only define the first half of the LSS. For LSS with odd symmetry, define the central ply with half of the thickness.

In [ ]:
LSS_symm = False

# L[layer_num] = [mat_num, orientation (deg), thickness (mm)]
L = np.array([
    [1,   0, 0.125],
    [1,  20, 0.125],
    [1, -20, 0.125],
    [1,  90, 0.125],
])


## Calculations

First it is necessary to complete the LSS in case a symmetric LSS is defined:

In [ ]:
n_lam = L.shape[0]

if LSS_symm:
    L = np.vstack([L, L[::-1]])

n_lam = L.shape[0]
L


#### Calculation and assembly of stiffness, compliance and transformation matrices

The compliance matrix S is calculated for each ply according to the definition for a transversely isotropic material. The stiffness matrix C is obtained by inverting S:

$$
S = \begin{bmatrix}
1/E_{11} & -\nu_{12}/E_{11} & -\nu_{12}/E_{11} & 0 & 0 & 0 \\
-\nu_{12}/E_{11} & 1/E_{22} & -\nu_{23}/E_{22} & 0 & 0 & 0 \\
-\nu_{12}/E_{11} & -\nu_{23}/E_{22} & 1/E_{22} & 0 & 0 & 0 \\
0 & 0 & 0 & 2(1+\nu_{23})/E_{22} & 0 & 0 \\
0 & 0 & 0 & 0 & 1/G_{12} & 0 \\
0 & 0 & 0 & 0 & 0 & 1/G_{12}
\end{bmatrix}
$$

In [ ]:
S = np.zeros((6, 6, n_lam))
C = np.zeros((6, 6, n_lam))

for i in range(n_lam):
    i_mat = int(L[i, 0])
    E11, E22, nu12, nu23, G12 = Mat[i_mat]

    S[0, 0, i] = 1 / E11
    S[0, 1, i] = -nu12 / E11
    S[0, 2, i] = -nu12 / E11
    S[1, 0, i] = S[0, 1, i]
    S[1, 1, i] = 1 / E22
    S[1, 2, i] = -nu23 / E22
    S[2, 0, i] = S[0, 2, i]
    S[2, 1, i] = S[1, 2, i]
    S[2, 2, i] = S[1, 1, i]
    S[3, 3, i] = 2 * (1 + nu23) / E22
    S[4, 4, i] = 1 / G12
    S[5, 5, i] = S[4, 4, i]

    C[:, :, i] = np.linalg.inv(S[:, :, i])


The transformation matrix T is calculated for each ply according to the definition. The transformation matrix for strain, `Tg`, is also obtained according to the definition.

In [ ]:
T = np.zeros((6, 6, n_lam))
Tg = np.zeros((6, 6, n_lam))

for i in range(n_lam):
    theta = L[i, 1] * np.pi / 180
    m = np.cos(theta)
    n = np.sin(theta)

    T[0, 0, i] = m**2
    T[0, 1, i] = n**2
    T[0, 5, i] = 2 * m * n
    T[1, 0, i] = n**2
    T[1, 1, i] = m**2
    T[1, 5, i] = -2 * m * n
    T[2, 2, i] = 1
    T[3, 3, i] = m
    T[3, 4, i] = -n
    T[4, 3, i] = n
    T[4, 4, i] = m
    T[5, 0, i] = -m * n
    T[5, 1, i] = m * n
    T[5, 5, i] = m**2 - n**2

    Tg[:, :, i] = np.linalg.inv(T[:, :, i]).T


The transformed compliance and stiffness matrices are obtained for each ply:

In [ ]:
Sb = np.zeros((6, 6, n_lam))
Cb = np.zeros((6, 6, n_lam))

for i in range(n_lam):
    Sb[:, :, i] = np.linalg.inv(Tg[:, :, i]) @ S[:, :, i] @ T[:, :, i]
    Cb[:, :, i] = np.linalg.inv(Sb[:, :, i])


### Calculation of the apparent stiffness and compliance matrices

The equivalent, or apparent, stiffness and compliance matrices of the laminate are calculated taking into account the relative thickness of each ply:

$$
\bar{C} = \sum_{i=1}^{n} \frac{t_i}{TH}\, \bar{C}_i \qquad \bar{S} = \bar{C}^{-1}
$$

In [ ]:
TH = L[:, 2].sum()

C_lam = np.zeros((6, 6))
for i in range(n_lam):
    C_lam += L[i, 2] / TH * Cb[:, :, i]

S_lam = np.linalg.inv(C_lam)

print("S_lam =")
print(S_lam)


In [ ]:
print("C_lam =")
print(C_lam)


### Calculation of the apparent engineering properties

The apparent engineering properties of the laminate are determined from the terms of the compliance matrix according to the definition:

$$
E_{xx} = \frac{1}{\bar{S}_{11}}, \quad
E_{yy} = \frac{1}{\bar{S}_{22}}, \quad
E_{zz} = \frac{1}{\bar{S}_{33}}, \quad
G_{yz} = \frac{1}{\bar{S}_{44}}, \quad
G_{zx} = \frac{1}{\bar{S}_{55}}, \quad
G_{xy} = \frac{1}{\bar{S}_{66}}
$$

$$
\nu_{xy} = -\frac{\bar{S}_{12}}{\bar{S}_{11}}, \quad
\nu_{yz} = -\frac{\bar{S}_{23}}{\bar{S}_{22}}, \quad
\nu_{zx} = -\frac{\bar{S}_{31}}{\bar{S}_{33}}
$$

### Results

**Note on units:** the input moduli (`Mat`) are given in MPa, and the code below (matching the
original MATLAB script exactly) does not convert to GPa. The column headers say "(GPa)" for
consistency with the original table, but the numeric values are actually in MPa, the same as in
the original script's output. Divide the modulus columns by 1000 if you want them genuinely in
GPa.

In [ ]:
Exx = 1 / S_lam[0, 0]
Eyy = 1 / S_lam[1, 1]
Ezz = 1 / S_lam[2, 2]
Gyz = 1 / S_lam[3, 3]
Gzx = 1 / S_lam[4, 4]
Gxy = 1 / S_lam[5, 5]
nuxy = -S_lam[0, 1] / S_lam[0, 0]
nuyz = -S_lam[1, 2] / S_lam[1, 1]
nuzx = -S_lam[2, 0] / S_lam[2, 2]

Results = pd.DataFrame(
    [[Exx, Eyy, Ezz, Gyz, Gzx, Gxy, nuxy, nuyz, nuzx]],
    columns=["Exx (GPa)", "Eyy (GPa)", "Ezz (GPa)", "Gyz (GPa)", "Gzx (GPa)",
             "Gxy (GPa)", "nuxy (-)", "nuyz (-)", "nuzx (-)"],
)
Results
